# Notebook 02 — LISTA, ALISTA & HyperLISTA: Training and Comparison

**Part A** — learned models on synthetic sparse recovery.

This notebook:
1. Trains LISTA end-to-end via BPTT
2. Trains ALISTA (analytic W, learnable γ, θ)
3. Tunes HyperLISTA via two-stage grid search
4. Plots NMSE (dB) vs. layer for all five methods
5. Produces a summary metrics table

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt

from src.data.sparse_generator import build_sparse_dataloaders
from src.models.ista       import ISTA
from src.models.fista      import FISTA
from src.models.lista      import LISTA
from src.models.alista     import ALISTA
from src.models.hyperlista import HyperLISTA
from src.training.trainer  import train
from src.training.tuner    import tune_hyperlista
from src.evaluation.metrics    import evaluate_model
from src.evaluation.visualizer import plot_nmse_vs_layers, get_layerwise_nmse

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_LAYERS = 16
print('Device:', DEVICE)

## 1. Data

In [ ]:
M, N, S = 250, 500, 50

A, train_loader, val_loader, test_loader = build_sparse_dataloaders(
    m=M, n=N, s=S, sigma=0.0,
    n_train=51200, n_val=2048, n_test=2048,
    batch_size=256, device=DEVICE,
)
print(f'Dataset ready. Train batches: {len(train_loader)}')

## 2. Classical Baselines (no training)

In [ ]:
ista  = ISTA(A, lam=0.1, n_iter=N_LAYERS).to(DEVICE)
fista = FISTA(A, lam=0.1, n_iter=N_LAYERS).to(DEVICE)

ista_nmse  = get_layerwise_nmse(ista,  test_loader, DEVICE)
fista_nmse = get_layerwise_nmse(fista, test_loader, DEVICE)

## 3. Train LISTA

In [ ]:
lista = LISTA(A, n_layers=N_LAYERS, tied=False).to(DEVICE)

history_lista = train(
    lista, train_loader, val_loader,
    n_epochs=50, lr=1e-3, device=DEVICE, patience=15, verbose=True,
)
lista_nmse = get_layerwise_nmse(lista, test_loader, DEVICE)
print(f'LISTA final NMSE: {lista_nmse[-1]:.2f} dB')

## 4. Train ALISTA

In [ ]:
# Note: ALISTA weight computation may take a few minutes
alista = ALISTA(A, n_layers=N_LAYERS, W_iters=2000).to(DEVICE)

history_alista = train(
    alista, train_loader, val_loader,
    n_epochs=50, lr=5e-4, device=DEVICE, patience=15, verbose=True,
)
alista_nmse = get_layerwise_nmse(alista, test_loader, DEVICE)
print(f'ALISTA final NMSE: {alista_nmse[-1]:.2f} dB')

## 5. Tune HyperLISTA

In [ ]:
hyperlista = HyperLISTA(A, n_layers=N_LAYERS, W_iters=2000).to(DEVICE)

tune_result = tune_hyperlista(
    hyperlista, val_loader, DEVICE,
    coarse_points=5, fine_points=7, verbose=True,
)
print('Tuned hyperparams:', tune_result)
hyperlista_nmse = get_layerwise_nmse(hyperlista, test_loader, DEVICE)
print(f'HyperLISTA final NMSE: {hyperlista_nmse[-1]:.2f} dB')

## 6. NMSE vs. Layer — All Methods

In [ ]:
all_results = {
    'ISTA':       ista_nmse,
    'FISTA':      fista_nmse,
    'LISTA':      lista_nmse,
    'ALISTA':     alista_nmse,
    'HyperLISTA': hyperlista_nmse,
}

fig = plot_nmse_vs_layers(
    all_results,
    title=f'NMSE vs. Layer  (m={M}, n={N}, s={S}, noiseless)',
)
fig.savefig('nmse_vs_layers_partA.pdf', bbox_inches='tight')
plt.show()

## 7. Summary Table

In [ ]:
import pandas as pd

rows = []
for name, model in [
    ('ISTA', ista), ('FISTA', fista),
    ('LISTA', lista), ('ALISTA', alista),
    ('HyperLISTA', hyperlista),
]:
    res = evaluate_model(model, test_loader, DEVICE)
    rows.append({'Model': name, **res})

df = pd.DataFrame(rows).set_index('Model')
print(df[['nmse_db', 'runtime_ms', 'n_params']].round(3).to_string())